<a href="https://colab.research.google.com/github/kunwar1908/Cognitive-Computing-UCS420/blob/main/Assignment_4_CC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420: Cognitive Computing
## Assignment 4 - A Cognitive FAQ System Using Pandas (Nova 2.0)

### Q1: Build Your Personalized Knowledge Base
- Extract the last two digits of your roll number[cite: 1].
- Compute category = `["billing", "account", "general"][d % 3]`[cite: 1].
- Build a 6-row DataFrame (4 fixed + 2 custom)[cite: 1].

In [7]:
import pandas as pd

ROLL_NUMBER = "1024170439"

categories_map = ["billing", "account", "general"]

d1 = int(ROLL_NUMBER[-2])
d2 = int(ROLL_NUMBER[-1])

cat1 = categories_map[d1 % 3]
cat2 = categories_map[d2 % 3]

print(f"Roll Number: {ROLL_NUMBER}")
print(f"Digit {d1} -> Category [{d1} % 3]: {cat1}")
print(f"Digit {d2} -> Category [{d2} % 3]: {cat2}\n")

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

custom_qa_pool = {
    "billing": {
        "question": "how can i download my invoice",
        "answer": "Navigate to Billing > Invoices and click Download PDF.",
        "keywords": "invoice bill receipt statement",
        "category": "billing"
    },
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Profile > Edit Info and verify via OTP to change number.",
        "keywords": "mobile phone number update profile",
        "category": "account"
    },
    "general": {
        "question": "where is the support office located",
        "answer": "Our main support office is located on the 2nd floor of Admin Block.",
        "keywords": "location address office campus",
        "category": "general"
    }
}

personalized_entries = [custom_qa_pool[cat1], custom_qa_pool[cat2]]

df_faq = pd.DataFrame(fixed_entries + personalized_entries)
df_faq

Roll Number: 1024170439
Digit 3 -> Category [3 % 3]: billing
Digit 9 -> Category [9 % 3]: billing



,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i download my invoice,Navigate to Billing > Invoices and click Downl...,invoice bill receipt statement,billing
5,how can i download my invoice,Navigate to Billing > Invoices and click Downl...,invoice bill receipt statement,billing


### Q2: Generate and Score a Hypothesis
Scoring function that takes a query string and returns matching entries ranked by confidence[cite: 1].

In [8]:
def score_query(query: str, df: pd.DataFrame) -> pd.DataFrame:
    query_tokens = set(query.lower().split())
    scores = []

    for _, row in df.iterrows():
        target_tokens = set(row["keywords"].lower().split() + row["question"].lower().split())
        match_count = len(query_tokens.intersection(target_tokens))
        scores.append(match_count)

    df_scored = df.copy()
    df_scored["score"] = scores
    return df_scored[df_scored["score"] > 0].sort_values(by="score", ascending=False)

test_query = "how to reset password and login"
score_query(test_query, df_faq)

,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,5
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1
4,how can i download my invoice,Navigate to Billing > Invoices and click Downl...,invoice bill receipt statement,billing,1
5,how can i download my invoice,Navigate to Billing > Invoices and click Downl...,invoice bill receipt statement,billing,1


### Q3: Questions in Same Category
Function `same_category(category_name, df)` returning all questions belonging to the assigned category[cite: 1].

In [9]:
def same_category(category_name: str, df: pd.DataFrame) -> pd.Series:
    return df[df["category"].str.lower() == category_name.lower()]["question"]

print(f"Questions in category '{cat1}':\n")
print(same_category(cat1, df_faq).to_string(index=False))

Questions in category 'billing':

       what is the annual fee
        how can i pay the fee
how can i download my invoice
how can i download my invoice


### Q4: Add Keyword and Save to CSV
Prompt the user for a new keyword, update an entry, and export to `<your_roll_number>_faq_data.csv`[cite: 1].

In [10]:
entry_index = 4
print(f"Target Entry Question: {df_faq.loc[entry_index, 'question']}")
print(f"Current Keywords: {df_faq.loc[entry_index, 'keywords']}")

new_keyword = input("Enter a new keyword to add: ").strip()
if new_keyword:
    df_faq.loc[entry_index, "keywords"] += f" {new_keyword}"
    print(f"Updated Keywords: {df_faq.loc[entry_index, 'keywords']}")

csv_filename = f"{ROLL_NUMBER}_faq_data.csv"
df_faq.to_csv(csv_filename, index=False)
print(f"Saved to {csv_filename}")

Target Entry Question: how can i download my invoice
Current Keywords: invoice bill receipt statement
Enter a new keyword to add: guns
Updated Keywords: invoice bill receipt statement guns
Saved to 1024170439_faq_data.csv


### Q5: FAQ Count Per Category (Groupby)
Count how many FAQ entries exist per category using `groupby`[cite: 1].

In [11]:
category_breakdown = df_faq.groupby("category")["question"].count().reset_index()
category_breakdown.rename(columns={"question": "faq_count"}, inplace=True)
category_breakdown

,category,faq_count
0,account,1
1,billing,4
2,general,1


### Q6: Tie-Aware Scoring Function
Prints all tied top entries rather than truncating[cite: 1].

In [12]:
def score_query_tie_aware(query: str, df: pd.DataFrame):
    ranked_df = score_query(query, df)
    print(f"\n=== Query: '{query}' ===")

    if ranked_df.empty:
        print("No matches found.")
        return

    max_score = ranked_df["score"].max()
    top_entries = ranked_df[ranked_df["score"] == max_score]

    if len(top_entries) > 1:
        print(f"[TIE DETECTED] {len(top_entries)} entries tied with top score ({max_score}):\n")
        for _, row in top_entries.iterrows():
            print(f"• Question: {row['question']}")
            print(f"  Answer:   {row['answer']} (Category: {row['category']})\n")
    else:
        print(f"[SINGLE TOP MATCH] Score: {max_score}\n")
        top = top_entries.iloc[0]
        print(f"• Question: {top['question']}")
        print(f"  Answer:   {top['answer']} (Category: {top['category']})")

score_query_tie_aware("how to pay the annual fee", df_faq)

score_query_tie_aware("working hours timing", df_faq)


=== Query: 'how to pay the annual fee' ===
[SINGLE TOP MATCH] Score: 4

• Question: how can i pay the fee
  Answer:   You can pay via UPI, card, or net banking. (Category: billing)

=== Query: 'working hours timing' ===
[SINGLE TOP MATCH] Score: 3

• Question: what are your working hours
  Answer:   We are open 9 AM to 5 PM. (Category: general)
